# 02 - Exploracao dos arquivos em `data`

Este notebook explora todos os arquivos encontrados em `data/`, com foco nos Parquets do dataset `spotify-metadata`.

Objetivos:
- listar os arquivos disponiveis e seus tamanhos;
- inspecionar schema, quantidade de linhas e amostras de cada Parquet;
- entender relacoes basicas entre tracks, artistas, albuns, playlists e audio features;
- abrir com cuidado arquivos JSON e JSONL compactados, sem carregar arquivos gigantes em memoria.

Os dados sao grandes. As celulas usam DuckDB e `LIMIT` sempre que possivel para evitar leitura completa desnecessaria.

## 0. Imports e configuracao

In [1]:
from __future__ import annotations

import io
import json
import shutil
import subprocess
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 160)

DATA_ROOT = Path("../data").resolve()
SPOTIFY_DIR = DATA_ROOT / "spotify-metadata"

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Pasta data nao encontrada: {DATA_ROOT}")

con = duckdb.connect()
print(f"DATA_ROOT = {DATA_ROOT}")

DATA_ROOT = D:\Mestrado\music-search-engine\data


## 1. Inventario dos arquivos

In [2]:
def format_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024


def classify_file(path: Path) -> str:
    name = path.name.lower()
    if name.endswith(".parquet"):
        return "parquet"
    if name.endswith(".jsonl.zst"):
        return "jsonl.zst"
    if name.endswith(".json"):
        return "json"
    if name.endswith(".zip"):
        return "zip"
    return path.suffix.lower().lstrip(".") or "sem extensao"


files = sorted(p for p in DATA_ROOT.rglob("*") if p.is_file() and p.name != ".gitkeep")

inventory = pd.DataFrame(
    [
        {
            "arquivo": str(path.relative_to(DATA_ROOT)).replace("\\", "/"),
            "tipo": classify_file(path),
            "pasta": str(path.parent.relative_to(DATA_ROOT)).replace("\\", "/"),
            "bytes": path.stat().st_size,
            "tamanho": format_bytes(path.stat().st_size),
        }
        for path in files
    ]
).sort_values(["tipo", "arquivo"])

display(inventory)
display(
    inventory.groupby("tipo", as_index=False)
    .agg(arquivos=("arquivo", "count"), bytes=("bytes", "sum"))
    .assign(tamanho_total=lambda d: d["bytes"].map(format_bytes))
    .sort_values("bytes", ascending=False)
)

,arquivo,tipo,pasta,bytes,tamanho
0,spotify-metadata/spotify_artist_redirects.json,json,spotify-metadata,282155,275.54 KB
1,spotify-metadata/spotify_audiobook_chapters.jsonl.zst,jsonl.zst,spotify-metadata,1553005565,1.45 GB
2,spotify-metadata/spotify_audiobooks.jsonl.zst,jsonl.zst,spotify-metadata,1757107274,1.64 GB
17,spotify-metadata/spotify_show_episodes.jsonl.zst,jsonl.zst,spotify-metadata,26964627558,25.11 GB
18,spotify-metadata/spotify_shows.jsonl.zst,jsonl.zst,spotify-metadata,39596247962,36.88 GB
3,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet,parquet,spotify-metadata/spotify_clean_audio_features_parquet,14122400248,13.15 GB
4,spotify-metadata/spotify_clean_parquet/album_images.parquet,parquet,spotify-metadata/spotify_clean_parquet,2566082417,2.39 GB
5,spotify-metadata/spotify_clean_parquet/albums.parquet,parquet,spotify-metadata/spotify_clean_parquet,4643662352,4.32 GB
6,spotify-metadata/spotify_clean_parquet/artist_albums.parquet,parquet,spotify-metadata/spotify_clean_parquet,666443131,635.57 MB
7,spotify-metadata/spotify_clean_parquet/artist_genres.parquet,parquet,spotify-metadata/spotify_clean_parquet,9965980,9.50 MB


,tipo,arquivos,bytes,tamanho_total
3,zip,1,158887514744,147.98 GB
2,parquet,14,120070064925,111.82 GB
1,jsonl.zst,4,69870988359,65.07 GB
0,json,1,282155,275.54 KB


## 2. Funcoes para explorar Parquets

Estas funcoes usam DuckDB para ler metadados, schema e pequenas amostras. Isso evita `pd.read_parquet()` direto em arquivos de muitos GB.

In [3]:
parquet_files = sorted(DATA_ROOT.rglob("*.parquet"))


def sql_path(path: Path) -> str:
    return str(path).replace("\\", "/").replace("'", "''")


def rel_path(path: Path) -> str:
    return str(path.relative_to(DATA_ROOT)).replace("\\", "/")


def parquet_schema(path: Path) -> pd.DataFrame:
    return con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{sql_path(path)}')"
    ).fetchdf()


def parquet_row_count(path: Path) -> int:
    return con.execute(
        f"SELECT count(*) AS n FROM read_parquet('{sql_path(path)}')"
    ).fetchone()[0]


def parquet_sample(path: Path, n: int = 5) -> pd.DataFrame:
    return con.execute(
        f"SELECT * FROM read_parquet('{sql_path(path)}') LIMIT {int(n)}"
    ).fetchdf()


def parquet_metadata_summary(path: Path) -> dict:
    try:
        meta = con.execute(
            f"SELECT * FROM parquet_metadata('{sql_path(path)}')"
        ).fetchdf()
    except Exception as exc:
        return {"row_groups": None, "metadata_error": str(exc)}

    out = {"row_groups": None, "metadata_error": None}
    if "row_group_id" in meta.columns:
        out["row_groups"] = int(meta["row_group_id"].nunique())
    if "stats_min" in meta.columns:
        out["columns_with_minmax"] = int(meta["stats_min"].notna().sum())
    return out


print(f"Parquets encontrados: {len(parquet_files)}")

Parquets encontrados: 14


## 3. Catalogo dos Parquets

In [4]:
catalog_rows = []

for path in parquet_files:
    schema = parquet_schema(path)
    meta = parquet_metadata_summary(path)
    catalog_rows.append(
        {
            "arquivo": rel_path(path),
            "linhas": parquet_row_count(path),
            "colunas": len(schema),
            "row_groups": meta.get("row_groups"),
            "tamanho": format_bytes(path.stat().st_size),
            "bytes": path.stat().st_size,
            "colunas_nome": ", ".join(schema["column_name"].astype(str).head(12)),
        }
    )

parquet_catalog = pd.DataFrame(catalog_rows).sort_values("bytes", ascending=False)
display(parquet_catalog.drop(columns=["bytes"]))

,arquivo,linhas,colunas,row_groups,tamanho,colunas_nome
13,spotify-metadata/spotify_clean_track_files_parquet/track_files.parquet,255966403,27,2084,49.26 GB,"rowid, track_id, filename, reencoded_kbit_vbr, fetched_at, session_country, sha256_original, sha256_with_embedded_meta, status, isrc_has_download, track_pop..."
9,spotify-metadata/spotify_clean_parquet/tracks.parquet,256039007,13,2084,22.13 GB,"rowid, id, fetched_at, name, preview_url, album_rowid, track_number, external_id_isrc, popularity, available_markets_rowid, disc_number, duration_ms"
11,spotify-metadata/spotify_clean_playlists_parquet/playlist_tracks.parquet,1698443099,15,13822,16.08 GB,"playlist_rowid, position, is_episode, track_rowid, id_if_not_in_tracks_table, added_at, added_by_id, primary_color, video_thumbnail_url, is_local, name_if_i..."
0,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet,255594909,17,2080,13.15 GB,"rowid, track_id, fetched_at, null_response, duration_ms, time_signature, tempo, key, mode, danceability, energy, loudness"
2,spotify-metadata/spotify_clean_parquet/albums.parquet,58590982,15,477,4.32 GB,"rowid, id, fetched_at, name, album_type, available_markets_rowid, external_id_upc, copyright_c, copyright_p, label, popularity, release_date"
1,spotify-metadata/spotify_clean_parquet/album_images.parquet,175809992,4,1431,2.39 GB,"album_rowid, width, height, url"
8,spotify-metadata/spotify_clean_parquet/track_artists.parquet,348055756,2,2833,1.63 GB,"track_rowid, artist_rowid"
12,spotify-metadata/spotify_clean_playlists_parquet/playlists.parquet,6608769,13,54,808.40 MB,"rowid, id, snapshot_id, fetched_at, name, description, collaborative, public, primary_color, owner_id, owner_display_name, followers_total"
3,spotify-metadata/spotify_clean_parquet/artist_albums.parquet,111808828,5,910,635.57 MB,"artist_rowid, album_rowid, is_appears_on, is_implicit_appears_on, index_in_album"
6,spotify-metadata/spotify_clean_parquet/artists.parquet,15430442,6,126,615.91 MB,"rowid, id, fetched_at, name, followers_total, popularity"


## 4. Schema de todos os Parquets

Use esta tabela para encontrar onde uma coluna aparece e comparar tipos entre arquivos.

In [5]:
schema_rows = []

for path in parquet_files:
    schema = parquet_schema(path)
    schema.insert(0, "arquivo", rel_path(path))
    schema_rows.append(schema)

all_schemas = pd.concat(schema_rows, ignore_index=True)
display(all_schemas)

display(
    all_schemas.groupby("column_name", as_index=False)
    .agg(
        ocorrencias=("arquivo", "count"),
        arquivos=("arquivo", lambda s: "; ".join(sorted(s.unique()))),
        tipos=("column_type", lambda s: ", ".join(sorted(map(str, s.unique())))),
    )
    .sort_values(["ocorrencias", "column_name"], ascending=[False, True])
)

,arquivo,column_name,column_type,null,key,default,extra
0,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet,rowid,VARCHAR,YES,None,None,None
1,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet,track_id,VARCHAR,YES,None,None,None
2,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet,fetched_at,VARCHAR,YES,None,None,None
3,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet,null_response,VARCHAR,YES,None,None,None
4,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet,duration_ms,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...,...
124,spotify-metadata/spotify_clean_track_files_parquet/track_files.parquet,original_title,VARCHAR,YES,None,None,None
125,spotify-metadata/spotify_clean_track_files_parquet/track_files.parquet,version_title,VARCHAR,YES,None,None,None
126,spotify-metadata/spotify_clean_track_files_parquet/track_files.parquet,file_id_mp3_96,VARCHAR,YES,None,None,None
127,spotify-metadata/spotify_clean_track_files_parquet/track_files.parquet,content_ratings,VARCHAR,YES,None,None,None


,column_name,ocorrencias,arquivos,tipos
70,rowid,7,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet; spotify-metadata/spotify_clean_parquet/albums.parquet; spotify-metadata/...,"BIGINT, VARCHAR"
26,fetched_at,6,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet; spotify-metadata/spotify_clean_parquet/albums.parquet; spotify-metadata/...,"BIGINT, VARCHAR"
8,artist_rowid,4,spotify-metadata/spotify_clean_parquet/artist_albums.parquet; spotify-metadata/spotify_clean_parquet/artist_genres.parquet; spotify-metadata/spotify_clean_p...,BIGINT
38,id,4,spotify-metadata/spotify_clean_parquet/albums.parquet; spotify-metadata/spotify_clean_parquet/artists.parquet; spotify-metadata/spotify_clean_parquet/tracks...,VARCHAR
54,name,4,spotify-metadata/spotify_clean_parquet/albums.parquet; spotify-metadata/spotify_clean_parquet/artists.parquet; spotify-metadata/spotify_clean_parquet/tracks...,VARCHAR
...,...,...,...,...
85,tracks_total,1,spotify-metadata/spotify_clean_playlists_parquet/playlists.parquet,BIGINT
86,uri_if_is_local,1,spotify-metadata/spotify_clean_playlists_parquet/playlist_tracks.parquet,VARCHAR
88,valence,1,spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet,VARCHAR
89,version_title,1,spotify-metadata/spotify_clean_track_files_parquet/track_files.parquet,VARCHAR


## 5. Amostra e schema de cada Parquet

Esta celula gera uma mini-secao para cada arquivo Parquet: schema completo e primeiras linhas.

In [6]:
for path in parquet_files:
    display(Markdown(f"### `{rel_path(path)}`"))
    display(Markdown(f"Tamanho: **{format_bytes(path.stat().st_size)}**"))
    display(parquet_schema(path))
    display(parquet_sample(path, n=5))

### `spotify-metadata/spotify_clean_audio_features_parquet/track_audio_features.parquet`

Tamanho: **13.15 GB**

,column_name,column_type,null,key,default,extra
0,rowid,VARCHAR,YES,None,None,None
1,track_id,VARCHAR,YES,None,None,None
2,fetched_at,VARCHAR,YES,None,None,None
3,null_response,VARCHAR,YES,None,None,None
4,duration_ms,VARCHAR,YES,None,None,None
5,time_signature,VARCHAR,YES,None,None,None
6,tempo,VARCHAR,YES,None,None,None
7,key,VARCHAR,YES,None,None,None
8,mode,VARCHAR,YES,None,None,None
9,danceability,VARCHAR,YES,None,None,None


,rowid,track_id,fetched_at,null_response,duration_ms,time_signature,tempo,key,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,1,2Pe9cbhOTvOUTDE4bl7zzl,1755734400000,0,630506,4,87.683,6,0,0.279,0.391,-12.054,0.32,0.816,0.737,0.177,0.0299
1,2,0wP732NKm8XgXu78XLRWoR,1755734400000,0,97216,4,105.298,5,1,0.429,0.318,-11.685,0.0566,0.587,0.782,0.202,0.36
2,3,22L6EJdnjx8oIo7GiF9hLe,1755734400000,0,75180,4,117.657,0,1,0.283,0.581,-9.42,0.0555,0.923,0.939,0.106,0.0362
3,4,3a519lgQ13JXNi0G73mwMT,1755734400000,0,149447,4,100.685,5,0,0.244,0.995,-0.69,0.125,0.78,0.799,0.132,0.0634
4,5,27yP7p2lxWYTtnldRN8Kzx,1755734400000,0,120816,4,123.499,7,1,0.313,0.618,0.411,0.073,0.843,0.109,0.126,0.187


### `spotify-metadata/spotify_clean_parquet/album_images.parquet`

...

,rowid,id,snapshot_id,fetched_at,name,description,collaborative,public,primary_color,owner_id,owner_display_name,followers_total,tracks_total
0,1,37i9dQZF1EP6YuccBxUcC1,AAAAAAAAAABuYPO2herY5rqaUQGREzFC,1741824000000,daylist,Your day in a playlist.,0,1,#ffffff,spotify,Spotify,0,0
1,2,37i9dQZF1EnOBYmteT8p3O,AAAAAAAAAAAhp7blsYnA3e5uTteKpTdV,1741824000000,Daily Podcasts,Podcast episodes picked just for you,0,1,#FFFFFF,spotify,Spotify,277840,11
2,3,37i9dQZF1DXcecv7ESbOPu,Z9RFUQAAAAAKqIKpDo7XUz37lI5PNCmA,1741824000000,New Music Friday Sweden,"Äntligen fredag och ny musik från Chappell Roan, Håkan Hellström, estraden och NOTD med flera. Happy New Music Friday!",0,1,#A0C3D2,spotify,Spotify,220102,104
3,4,37i9dQZF1DX3WvGXE8FqYX,Z8YrfAAAAAA5Mj8T6iA1UaVdXpVep4sn,1741824000000,Women of Pop,Celebrating the power of amazing female pop artists. Cover: Olivia Rodrigo & Lady Gaga,0,1,#ffffff,spotify,Spotify,2592601,75
4,5,37i9dQZF1DXc7FZ2VBjaeT,Z8y/QAAAAADh7aySbrlanbnlMPfJnbau,1741824000000,This Is Lady Gaga,"Listen to all her biggest hits, in one place.",0,1,#ffffff,spotify,Spotify,1784210,50


### `spotify-metadata/spotify_clean_track_files_parquet/track_files.parquet`

Tamanho: **49.26 GB**

,column_name,column_type,null,key,default,extra
0,rowid,BIGINT,YES,None,None,None
1,track_id,VARCHAR,YES,None,None,None
2,filename,VARCHAR,YES,None,None,None
3,reencoded_kbit_vbr,BIGINT,YES,None,None,None
4,fetched_at,BIGINT,YES,None,None,None
5,session_country,VARCHAR,YES,None,None,None
6,sha256_original,VARCHAR,YES,None,None,None
7,sha256_with_embedded_meta,VARCHAR,YES,None,None,None
8,status,VARCHAR,YES,None,None,None
9,isrc_has_download,BIGINT,YES,None,None,None


,rowid,track_id,filename,reencoded_kbit_vbr,fetched_at,session_country,sha256_original,sha256_with_embedded_meta,status,isrc_has_download,track_popularity,secondary_priority,prefixed_ogg_packet,alternatives,file_id_ogg_vorbis_96,file_id_ogg_vorbis_160,file_id_ogg_vorbis_320,file_id_aac_24,language_of_performance,artist_roles,has_lyrics,licensor,original_title,version_title,file_id_mp3_96,content_ratings,filesize_bytes
0,1892,003vvx7Niy0yvhvHt4a68B,track-popularity-50-to-100/T/TH/The Killers/2004 Hot Fuss (4piJq7R3gjUOxnYs6lDCTg)/02 The Killers - Mr. Brightside.ogg,<NA>,1741824000000,UNK,f4b083875794e0583cc686ab478b37f17eb5c392481b933d02a75763f9cc3dc3,f0ea763215eb83875dc84d00b7ea4357b561c0117af656987bc69e2a8c475530,success,<NA>,88,NaN,"[79, 103, 103, 83, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 179, 206, 40, 229, 1, 139, 129, 110, 0, 0, 164, 10, 150, 0, 151, 209, 62, 0, 1, 157...",None,e02d12f5e3540f96dc39e4de8baadeab0ab1dde7,d2ea54df9f2a62d53f2269ba0fee54875070ecbc,9ecc88fdbe8ae750528aaebcbf7ec9c219145402,3b6a4b4cf9f91dd5e21621ab185133b429d7b139,"[""en""]",None,<NA>,None,None,None,None,None,4258118
1,6955,00E0Z2jrF7reoHps4zcbWQ,track-popularity-50-to-100/A/AL/Alok/2023 Car Keys (Ayla) (1yUD0trOHc8dudwm9VAiHs)/1 Alok - Car Keys (Ayla).ogg,<NA>,1741824000000,UNK,eacb46fecfd3e8f3c34a7cd2fc1e072f5a07f8ced33ccafd52777a2f8f4a4b24,24bdc38cb51745c0cc08bb22220f94425c8041d0ea71fcfa65c649583fb03dcd,success,<NA>,71,NaN,"[79, 103, 103, 83, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 235, 206, 88, 2, 1, 139, 129, 110, 0, 0, 128, 15, 99, 0, 237, 106, 45, 0, 1, 128, 1...",None,6b811209ea1d1a5006b3845ca83850bc1ef72a78,6cf617e1df9db843c8dda75e501934706f63088a,b924f439dfe5959a9e9775ff8283ef69ec1e0fca,2bf0f011867c454e1cf6e7262c99bca1b8279767,"[""en""]",None,<NA>,None,None,None,None,None,3122766
2,7185,00GvqqIkMdHaxChyhZf9Nx,track-popularity-50-to-100/M/MA/Matroda/2024 4U (5p6wULtzOrjrTlMUtrDnVr)/1 Matroda - 4U.ogg,<NA>,1741824000000,UNK,e122d01a0dcc2fe725fc80b3bd475a66002eff0377838fbe736de8dd01159ab6,fccd174945a70da362b39e062888e5ce3d10de6bea43a4a1bcfa5b4c47ad12e4,success,<NA>,63,NaN,"[79, 103, 103, 83, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 101, 184, 79, 165, 1, 139, 129, 110, 0, 0, 239, 152, 121, 0, 3, 216, 51, 0, 1, 78, ...",None,92d1a2e0f598de67cb9a39025fc7ced6633cffa0,6d958060f849338883fcf1c0a6f4d81c00993c9a,42164b3518270ff2f8572a03343df88ec344870c,dc2a1bfc9715e12af8fb9c268fdd2f8a262ed2af,"[""zxx""]",None,<NA>,None,None,None,None,None,3465717
3,12733,003FTlCpBTM4eSqYSWPv4H,"track-popularity-50-to-100/T/TH/The All-American Rejects/2002 The All-American Rejects (0TvOeelcHQXYgPcyQiLhyR)/03 The All-American Rejects - Swing, Swing.ogg",<NA>,1741824000000,UNK,57c8314515bc72b7d10365729fc9ce8bf7c2e3c5176ee382d6f9da698f42b7b7,6b5fcc82d84f85851ecc60ab4784f29d0d6b5431dd9080df97c74fc08b74b71c,success,<NA>,68,NaN,"[79, 103, 103, 83, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 103, 108, 58, 210, 1, 139, 129, 110, 0, 0, 212, 247, 156, 0, 135, 240, 66, 0, 1, 69...",None,4c51cb1220d892c94d61b72c69e21e34e9806eae,691406af81c03022fd245be98f4c91de15269948,64b1bad3adb0ef2847c55b9427a8ef1f146d5c50,a7d681a75a47990236f695ab00ef65ce07e47b6e,"[""en""]",None,<NA>,None,None,None,None,None,4668038
4,20188,00AitwJWRhIH5IuNLGYLVD,track-popularity-28/C/CL/Clay Western/2024 When You Call (3RrkhujnaChJgMJzKS5Wbd)/1 Clay Western - When You Call.ogg,<NA>,1741824000000,UNK,1672a29c2ff3ef43cb2dd005692f435e87fb54920a4c6e8dbc712d1d3cd68e08,3cb0505192c99203abd305e84b5c7d6804ed4bf8069a5d270d09d05a72ac72a2,success,<NA>,28,NaN,"[79, 103, 103, 83, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 225, 111, 191, 203, 1, 139, 129, 110, 0, 0, 49, 115, 130, 0, 165, 157, 50, 0, 1, 14...",None,ffd33b4e2a25aedd930e3f2a9c1b477769d7fbe4,bac449a2060daf7f5ebe7d26f5d3f034469a1d81,1dd6a63b68b219ffde2c1da3ed912bfb15ac33d9,598a43e6fbd3cbb42721179bc27c554680f40aba,"[""en""]",None,<NA>,None,None,None,None,None,3466554


## 6. Perfil rapido de uma tabela selecionada

Altere `TABLE_KEY` para explorar uma tabela especifica com mais detalhe. A analise abaixo usa apenas uma amostra inicial para manter o notebook responsivo.

In [7]:
TABLE_KEY = "spotify_clean_parquet/tracks.parquet"  # altere aqui
SAMPLE_ROWS = 20_000

matches = [p for p in parquet_files if rel_path(p).endswith(TABLE_KEY)]
if not matches:
    raise ValueError(f"Nenhum parquet encontrado para TABLE_KEY={TABLE_KEY!r}")

table_path = matches[0]
display(Markdown(f"### Perfil de `{rel_path(table_path)}`"))

sample_df = con.execute(
    f"SELECT * FROM read_parquet('{sql_path(table_path)}') LIMIT {SAMPLE_ROWS}"
).fetchdf()

display(sample_df.head(10))
display(
    pd.DataFrame(
        {
            "dtype": sample_df.dtypes.astype(str),
            "nulos_amostra": sample_df.isna().sum(),
            "nulos_%_amostra": (sample_df.isna().mean() * 100).round(2),
            "unicos_amostra": sample_df.nunique(dropna=True),
        }
    )
)
display(sample_df.describe(include="all").T)

### Perfil de `spotify-metadata/spotify_clean_parquet/tracks.parquet`

,rowid,id,fetched_at,name,preview_url,album_rowid,track_number,external_id_isrc,popularity,available_markets_rowid,disc_number,duration_ms,explicit
0,1,5xHgo5JN0wfsV41HnRaos5,1743033600000,The Giver,NaN,1,1,USUG12501598,89,2,1,202768,0
1,2,4kcRyBdCtBxcq14yDzVjJ0,1743033600000,Another Life,NaN,2,1,SEUM72401960,43,2,1,139388,0
2,3,2x1TExAkrUFFxVk616CKw8,1743033600000,Lover Online,NaN,2,2,SEUM72401353,43,2,1,167444,0
3,4,0Hf3fR6XKINmMB4Fey7TiH,1743033600000,Crash,NaN,2,3,SEUM72500201,56,2,1,147601,0
4,5,0nWMjJ0b226HZP139cPKqw,1743033600000,I Just Missed A Call,NaN,2,4,SEUM72500202,51,2,1,146424,0
5,6,56T5Dg1rFBiNOK928y00w2,1743033600000,Bruce Wayne,NaN,2,5,SEUM72401497,41,2,1,131203,0
6,7,6gYTsbzxbuqHAFopB5tCrQ,1743033600000,WIFI,NaN,2,6,SEUM72500203,55,2,1,146659,0
7,8,7abZdMxSDfDDf7HKB8Ae8r,1743033600000,SMOKE THE PAIN AWAY,https://p.scdn.co/mp3-preview/a24ec01c02718e82904ead64dc6bbd733c194528?cid=65b708073fc0480ea92a077233ca87bd,3,1,GBARL2500153,75,3,1,162772,0
8,9,4QIzYCGDTLJxue9HQe4xaY,1741824000000,Leyla,https://p.scdn.co/mp3-preview/31e06a183ebdde4b931737558c073e2655bf5853?cid=65b708073fc0480ea92a077233ca87bd,4,1,US38Y2510866,45,4,1,146352,1
9,10,2pUjzh95Vc8GBCD5cRn3lb,1741824000000,Varningsklocka,https://p.scdn.co/mp3-preview/bae0f73fdad0dc3dcf2125a8069d3dbf31698b13?cid=65b708073fc0480ea92a077233ca87bd,5,1,SEBGA2500341,43,3,1,170000,0


,dtype,nulos_amostra,nulos_%_amostra,unicos_amostra
rowid,int64,0,0.0,20000
id,str,0,0.0,20000
fetched_at,int64,0,0.0,8
name,str,0,0.0,17835
preview_url,str,7840,39.2,12043
album_rowid,int64,0,0.0,2211
track_number,int64,0,0.0,100
external_id_isrc,str,0,0.0,19181
popularity,int64,0,0.0,99
available_markets_rowid,int64,0,0.0,244


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
rowid,20000.0,NaN,NaN,NaN,10000.5,5773.647028,1.0,5000.75,10000.5,15000.25,20000.0
id,20000,20000,5xHgo5JN0wfsV41HnRaos5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fetched_at,20000.0,NaN,NaN,NaN,1743082014240.0,466665720.328748,1741824000000.0,1743033600000.0,1743033600000.0,1743033600000.0,1746057600000.0
name,20000,17835,Intro,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
preview_url,12160,12043,https://p.scdn.co/mp3-preview/ed130cf84883116b96794ecf2bb9e29a944932a8?cid=65b708073fc0480ea92a077233ca87bd,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
album_rowid,20000.0,NaN,NaN,NaN,1198.9342,583.444171,1.0,773.0,1297.0,1666.0,2211.0
track_number,20000.0,NaN,NaN,NaN,7.77265,6.641345,1.0,3.0,7.0,11.0,100.0
external_id_isrc,20000,19181,USUM70807646,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
popularity,20000.0,NaN,NaN,NaN,41.511,21.185514,0.0,26.0,40.0,57.0,100.0
available_markets_rowid,20000.0,NaN,NaN,NaN,26.48645,52.925007,1.0,3.0,3.0,8.0,247.0


## 7. Exemplos de relacoes entre tabelas

As tabelas normalizadas se conectam principalmente por `rowid`, `*_rowid` e IDs publicos do Spotify.

In [8]:
CLEAN_DIR = SPOTIFY_DIR / "spotify_clean_parquet"
AUDIO_DIR = SPOTIFY_DIR / "spotify_clean_audio_features_parquet"
PLAYLIST_DIR = SPOTIFY_DIR / "spotify_clean_playlists_parquet"

paths = {
    "tracks": CLEAN_DIR / "tracks.parquet",
    "track_artists": CLEAN_DIR / "track_artists.parquet",
    "artists": CLEAN_DIR / "artists.parquet",
    "albums": CLEAN_DIR / "albums.parquet",
    "artist_genres": CLEAN_DIR / "artist_genres.parquet",
    "audio_features": AUDIO_DIR / "track_audio_features.parquet",
    "playlists": PLAYLIST_DIR / "playlists.parquet",
    "playlist_tracks": PLAYLIST_DIR / "playlist_tracks.parquet",
}

for key, path in paths.items():
    print(f"{key:16s}", "OK" if path.exists() else "NAO ENCONTRADO", path)

tracks           OK D:\Mestrado\music-search-engine\data\spotify-metadata\spotify_clean_parquet\tracks.parquet
track_artists    OK D:\Mestrado\music-search-engine\data\spotify-metadata\spotify_clean_parquet\track_artists.parquet
artists          OK D:\Mestrado\music-search-engine\data\spotify-metadata\spotify_clean_parquet\artists.parquet
albums           OK D:\Mestrado\music-search-engine\data\spotify-metadata\spotify_clean_parquet\albums.parquet
artist_genres    OK D:\Mestrado\music-search-engine\data\spotify-metadata\spotify_clean_parquet\artist_genres.parquet
audio_features   OK D:\Mestrado\music-search-engine\data\spotify-metadata\spotify_clean_audio_features_parquet\track_audio_features.parquet
playlists        OK D:\Mestrado\music-search-engine\data\spotify-metadata\spotify_clean_playlists_parquet\playlists.parquet
playlist_tracks  OK D:\Mestrado\music-search-engine\data\spotify-metadata\spotify_clean_playlists_parquet\playlist_tracks.parquet


In [9]:
# Tracks + album + primeiro artista + audio features
query = f"""
WITH first_artist AS (
    SELECT track_rowid, min(artist_rowid) AS artist_rowid
    FROM read_parquet('{sql_path(paths["track_artists"])}')
    GROUP BY track_rowid
)
SELECT
    t.rowid AS track_rowid,
    t.id AS track_id,
    t.name AS track_name,
    ar.name AS artist_name,
    alb.name AS album_name,
    t.popularity,
    t.duration_ms,
    af.danceability,
    af.energy,
    af.valence,
    af.tempo
FROM read_parquet('{sql_path(paths["tracks"])}') AS t
LEFT JOIN first_artist fa ON fa.track_rowid = t.rowid
LEFT JOIN read_parquet('{sql_path(paths["artists"])}') AS ar ON ar.rowid = fa.artist_rowid
LEFT JOIN read_parquet('{sql_path(paths["albums"])}') AS alb ON alb.rowid = t.album_rowid
LEFT JOIN read_parquet('{sql_path(paths["audio_features"])}') AS af ON af.track_id = t.id
LIMIT 20
"""

display(con.execute(query).fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,track_rowid,track_id,track_name,artist_name,album_name,popularity,duration_ms,danceability,energy,valence,tempo
0,139140965,0brO7dnKO85JiSrHD4QhRK,NBK,Claud Chaos,Into the Void,0,245783,0.45,0.92,0.35,144.954
1,139140970,5ANmB7pNUJbrDFMhCNzaGI,Found My Home,Lucie Tiger,Alabama Highway,1,198986,0.623,0.79,0.638,91.079
2,139140973,4EIYE8f2nxkd4Py3V3RWSW,Midnight Goodbye,Lucie Tiger,Alabama Highway,0,254200,0.704,0.683,0.595,140.004
3,139140976,6dUZfOYGNnL6JNWYvIY9sW,Start Again,L E O,Dreams,0,349200,0.671,0.714,0.509,128.05
4,139140978,3QspTiPM24f648ArFvMjIn,Easy,L E O,Dreams,0,333750,0.627,0.919,0.452,128.04
5,139140981,37prwfrk31lKFppTvsF9SU,Closer,L E O,Dreams,0,273750,0.643,0.833,0.579,127.998
6,139140982,4f4eQMDftUTj6bve773THq,Sirens,L E O,Dreams,0,335238,0.683,0.893,0.602,128.007
7,139140984,6CaogQpg5qk2YbQFH83VUt,Valiant,L E O,Silence,0,240000,0.732,0.746,0.457,123.01
8,139140989,6JzXZmj7ySKO4egREHcXtT,Silence,L E O,Silence,2,266666,0.583,0.874,0.827,126.053
9,139140994,7GbaAvYKgVtr33vz1mnFHS,Light,L E O,Silence,0,326250,0.666,0.856,0.169,127.994


In [10]:
# Playlists + tracks
# As colunas exatas podem variar; se der erro, rode parquet_schema(paths["playlist_tracks"]) para ajustar.
display(parquet_schema(paths["playlists"]))
display(parquet_schema(paths["playlist_tracks"]))
display(parquet_sample(paths["playlists"], 10))
display(parquet_sample(paths["playlist_tracks"], 10))

,column_name,column_type,null,key,default,extra
0,rowid,BIGINT,YES,None,None,None
1,id,VARCHAR,YES,None,None,None
2,snapshot_id,VARCHAR,YES,None,None,None
3,fetched_at,BIGINT,YES,None,None,None
4,name,VARCHAR,YES,None,None,None
5,description,VARCHAR,YES,None,None,None
6,collaborative,BIGINT,YES,None,None,None
7,public,BIGINT,YES,None,None,None
8,primary_color,VARCHAR,YES,None,None,None
9,owner_id,VARCHAR,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,playlist_rowid,BIGINT,YES,None,None,None
1,position,BIGINT,YES,None,None,None
2,is_episode,BIGINT,YES,None,None,None
3,track_rowid,BIGINT,YES,None,None,None
4,id_if_not_in_tracks_table,VARCHAR,YES,None,None,None
5,added_at,BIGINT,YES,None,None,None
6,added_by_id,VARCHAR,YES,None,None,None
7,primary_color,VARCHAR,YES,None,None,None
8,video_thumbnail_url,VARCHAR,YES,None,None,None
9,is_local,BIGINT,YES,None,None,None


,rowid,id,snapshot_id,fetched_at,name,description,collaborative,public,primary_color,owner_id,owner_display_name,followers_total,tracks_total
0,1,37i9dQZF1EP6YuccBxUcC1,AAAAAAAAAABuYPO2herY5rqaUQGREzFC,1741824000000,daylist,Your day in a playlist.,0,1,#ffffff,spotify,Spotify,0,0
1,2,37i9dQZF1EnOBYmteT8p3O,AAAAAAAAAAAhp7blsYnA3e5uTteKpTdV,1741824000000,Daily Podcasts,Podcast episodes picked just for you,0,1,#FFFFFF,spotify,Spotify,277840,11
2,3,37i9dQZF1DXcecv7ESbOPu,Z9RFUQAAAAAKqIKpDo7XUz37lI5PNCmA,1741824000000,New Music Friday Sweden,"Äntligen fredag och ny musik från Chappell Roan, Håkan Hellström, estraden och NOTD med flera. Happy New Music Friday!",0,1,#A0C3D2,spotify,Spotify,220102,104
3,4,37i9dQZF1DX3WvGXE8FqYX,Z8YrfAAAAAA5Mj8T6iA1UaVdXpVep4sn,1741824000000,Women of Pop,Celebrating the power of amazing female pop artists. Cover: Olivia Rodrigo & Lady Gaga,0,1,#ffffff,spotify,Spotify,2592601,75
4,5,37i9dQZF1DXc7FZ2VBjaeT,Z8y/QAAAAADh7aySbrlanbnlMPfJnbau,1741824000000,This Is Lady Gaga,"Listen to all her biggest hits, in one place.",0,1,#ffffff,spotify,Spotify,1784210,50
5,6,37i9dQZF1DWVRbdKqFtIef,Z9OpwQAAAAA9Ks6HKi8d+0a7dd7/iFvC,1741824000000,This is Chappell Roan,This is Chappell Roan. The essential tracks all in one playlist.,0,1,#ffffff,spotify,Spotify,232151,23
6,7,37i9dQZF1DZ06evO4iRboc,Z9NxgAAAAABiIqB0g2jGV9pM7HNR0vLC,1741824000000,This Is Laufey,"This is Laufey. The essential tracks, all in one playlist.",0,1,NaN,spotify,Spotify,268233,32
7,8,37i9dQZF1DX1PfYnYcpw8w,Z2+SJwAAAAB0M85xBLVgDGID5oh4dSEt,1741824000000,This Is Ariana Grande,"The essential tracks, all in one playlist.",0,1,#ffffff,spotify,Spotify,3686877,75
8,9,37i9dQZF1DX5KpP2LN299J,Z7SyNQAAAAAJtFJ0zG1UG43zZ3xLkbou,1741824000000,This Is Taylor Swift,"The essential tracks, all in one playlist.",0,1,#FFFFFF,spotify,Spotify,6173059,179
9,10,37i9dQZF1DZ06evO2Cuzya,Z9NxgAAAAACBCIw9UBo5kGAcfCnqgTov,1741824000000,This Is Gracie Abrams,"This is Gracie Abrams. The essential tracks, all in one playlist.",0,1,NaN,spotify,Spotify,174565,39


,playlist_rowid,position,is_episode,track_rowid,id_if_not_in_tracks_table,added_at,added_by_id,primary_color,video_thumbnail_url,is_local,name_if_is_local,uri_if_is_local,album_name_if_is_local,artists_name_if_is_local,duration_ms_if_is_local
0,2,0,1,<NA>,5SeY2D5YVJ1I4hmzGOtVsk,0,,None,None,0,None,None,None,None,<NA>
1,2,1,0,<NA>,NaN,0,,None,None,0,None,None,None,None,<NA>
2,2,2,1,<NA>,3cCPgGLmgJqp8k3uQvUfD7,0,,None,None,0,None,None,None,None,<NA>
3,2,3,1,<NA>,2zT28jtBbq3k9h5vGEAm6M,0,,None,None,0,None,None,None,None,<NA>
4,2,4,1,<NA>,5nVUSVNCbzYYLXIZ40dNci,0,,None,None,0,None,None,None,None,<NA>
5,2,5,1,<NA>,4Re9dFl7hKSkgViOMSsgdF,0,,None,None,0,None,None,None,None,<NA>
6,2,6,1,<NA>,7DBiBnYPLiUF2gSwrbNaxw,0,,None,None,0,None,None,None,None,<NA>
7,2,7,1,<NA>,1ao65nUsojlTqHch0o6FDp,0,,None,None,0,None,None,None,None,<NA>
8,2,8,1,<NA>,4H5OCAvlQnxciZPhThvrS8,0,,None,None,0,None,None,None,None,<NA>
9,2,9,1,<NA>,3GbHlWzgaO9zT108n5Nvnm,0,,None,None,0,None,None,None,None,<NA>


## 8. Arquivo JSON pequeno: redirects de artistas

In [11]:
redirects_path = SPOTIFY_DIR / "spotify_artist_redirects.json"

if redirects_path.exists():
    with redirects_path.open("r", encoding="utf-8") as f:
        redirects = json.load(f)

    print(type(redirects), "len=", len(redirects) if hasattr(redirects, "__len__") else "n/a")

    if isinstance(redirects, dict):
        display(pd.DataFrame(list(redirects.items())[:20], columns=["origem", "destino"]))
    elif isinstance(redirects, list):
        display(pd.DataFrame(redirects[:20]))
    else:
        display(redirects)
else:
    print("Arquivo nao encontrado:", redirects_path)

<class 'list'> len= 3974


,from_id,to_id
0,44gXJLzX8iq8Ie4wjBDWsw,1CaCR4aY8KkwBTIVgAsl2L
1,6LfRKKiOHyxMuei2Pn3kMl,540vIaP2JwjQb9dm3aArA4
2,6x6Gqn5iSdTEcUZAdJ2UId,540vIaP2JwjQb9dm3aArA4
3,2TQm48ajZxZ3u4KNVckeCr,3PhoLpVuITZKcymswpck5b
4,3d8tIMxIqNVG58a0ey5ato,3PhoLpVuITZKcymswpck5b
5,2uqgtwXCqZ9qhvnmnM4jyM,1QIoedC1hO9QOSfQfFuLeI
6,0Ou5Z7WQUrz6uWquJBHnaS,3nFkdlSjzX9mRTtwJOzDYB
7,6jEQFo3iBSyW1f9DVBLknb,3nFkdlSjzX9mRTtwJOzDYB
8,6zW6zGP27cszsSZ1tGY17p,1Xyo4u8uXC1ZmMpatF05PJ
9,0TmlxNKjCPRpUOs1nMyaEL,0EmeFodog0BfCgMzAIvKQp


## 9. Amostras de JSONL compactado (`.jsonl.zst`)

Os arquivos `.jsonl.zst` sao muito grandes. Esta secao tenta ler apenas as primeiras linhas.

Ordem de tentativa:
1. pacote Python `zstandard`, se estiver instalado;
2. executavel `zstd`, se existir no sistema;
3. DuckDB `read_json_auto`, como fallback.

Se nenhuma opcao funcionar, instale o leitor de zstd no ambiente do projeto, por exemplo: `uv add --dev zstandard`.

In [12]:
jsonl_zst_files = sorted(DATA_ROOT.rglob("*.jsonl.zst"))
display(pd.DataFrame({"arquivo": [rel_path(p) for p in jsonl_zst_files], "tamanho": [format_bytes(p.stat().st_size) for p in jsonl_zst_files]}))


def read_jsonl_zst_head(path: Path, n: int = 5) -> pd.DataFrame:
    rows = []

    try:
        import zstandard as zstd

        with path.open("rb") as fh:
            reader = zstd.ZstdDecompressor().stream_reader(fh)
            text_stream = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")
            for line in text_stream:
                if line.strip():
                    rows.append(json.loads(line))
                if len(rows) >= n:
                    break
        return pd.DataFrame(rows)
    except Exception as exc_python:
        python_error = exc_python

    zstd_exe = shutil.which("zstd")
    if zstd_exe:
        try:
            proc = subprocess.Popen(
                [zstd_exe, "-dc", "-q", str(path)],
                text=True,
                encoding="utf-8",
                errors="replace",
                stdout=subprocess.PIPE,
                stderr=subprocess.DEVNULL,
            )
            assert proc.stdout is not None
            try:
                for line in proc.stdout:
                    if line.strip():
                        rows.append(json.loads(line))
                    if len(rows) >= n:
                        break
            finally:
                proc.kill()
                proc.wait(timeout=5)
            if rows:
                return pd.DataFrame(rows)
        except Exception as exc_cli:
            cli_error = exc_cli
    else:
        cli_error = RuntimeError("executavel zstd nao encontrado")

    try:
        return con.execute(
            f"SELECT * FROM read_json_auto('{sql_path(path)}', maximum_object_size=16777216) LIMIT {int(n)}"
        ).fetchdf()
    except Exception as exc_duckdb:
        raise RuntimeError(
            "Nao consegui ler amostra do JSONL.ZST. "
            f"python={python_error}; cli={cli_error}; duckdb={exc_duckdb}"
        )


for path in jsonl_zst_files:
    display(Markdown(f"### `{rel_path(path)}`"))
    try:
        display(read_jsonl_zst_head(path, n=3))
    except Exception as exc:
        print(exc)

,arquivo,tamanho
0,spotify-metadata/spotify_audiobook_chapters.jsonl.zst,1.45 GB
1,spotify-metadata/spotify_audiobooks.jsonl.zst,1.64 GB
2,spotify-metadata/spotify_show_episodes.jsonl.zst,25.11 GB
3,spotify-metadata/spotify_shows.jsonl.zst,36.88 GB


### `spotify-metadata/spotify_audiobook_chapters.jsonl.zst`

,id,description,chapter_number,duration_ms,images,languages,name,audio_preview_url,release_date,release_date_precision,explicit,html_description,available_markets,type,uri,external_urls,href,audiobook
0,5nHj2BqY4EY5J7ngTwgWXN,,0,15916,"[{'height': 640, 'width': 640, 'url': 'https://i.scdn.co/image/ab676663000022a86a1e3c8df850ec21e3c78b15'}, {'height': 300, 'width': 300, 'url': 'https://i.s...",[],Intro,None,2021-09-29,day,False,,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...",chapter,spotify:episode:5nHj2BqY4EY5J7ngTwgWXN,{'spotify': 'https://open.spotify.com/episode/5nHj2BqY4EY5J7ngTwgWXN'},https://api.spotify.com/v1/chapters/5nHj2BqY4EY5J7ngTwgWXN,"{'authors': [{'name': 'Blake Pierce'}], 'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CA', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE'..."
1,6VqMlSpSQDUVb1telDlSsk,,1,835511,"[{'height': 640, 'width': 640, 'url': 'https://i.scdn.co/image/ab676663000022a86a1e3c8df850ec21e3c78b15'}, {'height': 300, 'width': 300, 'url': 'https://i.s...",[],Chapter 1,None,2021-09-29,day,False,,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...",chapter,spotify:episode:6VqMlSpSQDUVb1telDlSsk,{'spotify': 'https://open.spotify.com/episode/6VqMlSpSQDUVb1telDlSsk'},https://api.spotify.com/v1/chapters/6VqMlSpSQDUVb1telDlSsk,"{'authors': [{'name': 'Blake Pierce'}], 'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CA', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE'..."
2,25T14Iuul2YSRHXy1lLOPy,,2,661319,"[{'height': 640, 'width': 640, 'url': 'https://i.scdn.co/image/ab676663000022a86a1e3c8df850ec21e3c78b15'}, {'height': 300, 'width': 300, 'url': 'https://i.s...",[],Chapter 2,None,2021-09-29,day,False,,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...",chapter,spotify:episode:25T14Iuul2YSRHXy1lLOPy,{'spotify': 'https://open.spotify.com/episode/25T14Iuul2YSRHXy1lLOPy'},https://api.spotify.com/v1/chapters/25T14Iuul2YSRHXy1lLOPy,"{'authors': [{'name': 'Blake Pierce'}], 'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CA', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE'..."


### `spotify-metadata/spotify_audiobooks.jsonl.zst`

,authors,available_markets,chapters,copyrights,description,edition,external_urls,explicit,href,html_description,id,images,languages,media_type,name,narrators,publisher,total_chapters,type,uri
0,[{'name': 'Blake Pierce'}],"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...","{'total': 41, 'items': [{'id': '5nHj2BqY4EY5J7ngTwgWXN', 'description': '', 'chapter_number': 0, 'duration_ms': 15916, 'images': [{'height': 640, 'width': 6...",[],"Author(s): Blake Pierce\nNarrator(s): Rachael Caise\n\n“When you think that life cannot get better, Blake Pierce comes up with another masterpiece of thrill...",Unabridged,{'spotify': 'https://open.spotify.com/show/7g94ZuU03O7H8rHXRjaKcj'},False,https://api.spotify.com/v1/audiobooks/7g94ZuU03O7H8rHXRjaKcj?locale=*,"Author(s): Blake Pierce<br/>Narrator(s): Rachael Caise<br/>“When you think that life cannot get better, Blake Pierce comes up with another masterpiece of th...",7g94ZuU03O7H8rHXRjaKcj,"[{'url': 'https://i.scdn.co/image/ab676663000022a86a1e3c8df850ec21e3c78b15', 'height': 640, 'width': 640}, {'url': 'https://i.scdn.co/image/ab6766630000db5b...",[en],audio,Left to Fear (An Adele Sharp Mystery—Book Ten),[{'name': 'Rachael Caise'}],Blake Pierce,41,audiobook,spotify:show:7g94ZuU03O7H8rHXRjaKcj
1,[{'name': 'Blake Pierce'}],"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...","{'total': 69, 'items': [{'id': '3Av1bAao461aLovrXpnnME', 'description': '', 'chapter_number': 0, 'duration_ms': 25798, 'images': [{'height': 640, 'width': 6...",[],Author(s): Blake Pierce\nNarrator(s): Abigail Reno\n\nA bundle of books #1 (LEFT TO DIE) and #2 (LEFT TO RUN) in Blake Pierce’s Adele Sharp Mystery series! ...,Unabridged,{'spotify': 'https://open.spotify.com/show/3KhaMr3tewziJcLH7KFaC0'},False,https://api.spotify.com/v1/audiobooks/3KhaMr3tewziJcLH7KFaC0?locale=*,Author(s): Blake Pierce<br/>Narrator(s): Abigail Reno<br/>A bundle of books #1 (LEFT TO DIE) and #2 (LEFT TO RUN) in Blake Pierce’s Adele Sharp Mystery seri...,3KhaMr3tewziJcLH7KFaC0,"[{'url': 'https://i.scdn.co/image/ab676663000022a80848eac52f105281d11b3038', 'height': 640, 'width': 640}, {'url': 'https://i.scdn.co/image/ab6766630000db5b...",[en],audio,An Adele Sharp Mystery Bundle: Left to Die (#1) and Left to Run (#2),[{'name': 'Abigail Reno'}],Blake Pierce,69,audiobook,spotify:show:3KhaMr3tewziJcLH7KFaC0
2,[{'name': 'Blake Pierce'}],"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...","{'total': 65, 'items': [{'id': '0YETCXfqxreDhaLTeddPNR', 'description': '', 'chapter_number': 0, 'duration_ms': 16692, 'images': [{'height': 640, 'width': 6...",[],Author(s): Blake Pierce\nNarrator(s): Abigail Reno\n\nA bundle of books #4 (LEFT TO KILL) and #5 (LEFT TO MURDER) in Blake Pierce’s Adele Sharp Mystery seri...,Unabridged,{'spotify': 'https://open.spotify.com/show/5BTjabSt7kJKDyop3tWqib'},False,https://api.spotify.com/v1/audiobooks/5BTjabSt7kJKDyop3tWqib?locale=*,Author(s): Blake Pierce<br/>Narrator(s): Abigail Reno<br/>A bundle of books #4 (LEFT TO KILL) and #5 (LEFT TO MURDER) in Blake Pierce’s Adele Sharp Mystery ...,5BTjabSt7kJKDyop3tWqib,"[{'url': 'https://i.scdn.co/image/ab676663000022a8d2b49eadd1ae7312cbfc6a03', 'height': 640, 'width': 640}, {'url': 'https://i.scdn.co/image/ab6766630000db5b...",[en],audio,An Adele Sharp Mystery Bundle: Left to Kill (#4) and Left to Murder (#5),[{'name': 'Abigail Reno'}],Blake Pierce,65,audiobook,spotify:show:5BTjabSt7kJKDyop3tWqib


### `spotify-metadata/spotify_show_episodes.jsonl.zst`

,explicit,audio_preview_url,description,duration_ms,episode,external_urls,href,html_description,id,images,is_externally_hosted,language,languages,name,release_date,release_date_precision,available_markets,show,track,type,uri,is_playable,restrictions
0,False,https://podz-content.spotifycdn.com/audio/clips/0MBzsMJP0DPQTyPq0Cugth/clip_0_5616.mp3,"The more you listen, the more personalized it gets",5616,True,{'spotify': 'https://open.spotify.com/episode/5SeY2D5YVJ1I4hmzGOtVsk'},https://api.spotify.com/v1/episodes/5SeY2D5YVJ1I4hmzGOtVsk,"The more you listen, the more personalized it gets",5SeY2D5YVJ1I4hmzGOtVsk,"[{'height': 640, 'url': 'https://i.scdn.co/image/ab6765630000ba8acbf16b4ccfd8be044fbf6d3e', 'width': 640}, {'height': 300, 'url': 'https://i.scdn.co/image/a...",False,en,[en],Get started listening to podcasts,2019-11-18,day,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...","{'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CA', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE', 'EC', 'EE', 'SV', 'FI', 'FR', 'GR', '...",False,episode,spotify:episode:5SeY2D5YVJ1I4hmzGOtVsk,<NA>,<NA>
1,False,https://podz-content.spotifycdn.com/audio/clips/04jKRc3UfQQBZzNQAGrhnv/clip_0_60000.mp3,"Har du inte blivit del av Mellan Himmel och Jord+ än? Här får du ett smakprov på avsnitt två av bonuspodden Naket och Nära, där vi hjälper er lyssnare med e...",168542,True,{'spotify': 'https://open.spotify.com/episode/3cCPgGLmgJqp8k3uQvUfD7'},https://api.spotify.com/v1/episodes/3cCPgGLmgJqp8k3uQvUfD7,"<p>Har du inte blivit del av Mellan Himmel och Jord&#43; än? Här får du ett smakprov på avsnitt två av bonuspodden Naket och Nära, där vi hjälper er lyssnar...",3cCPgGLmgJqp8k3uQvUfD7,"[{'height': 640, 'url': 'https://i.scdn.co/image/ab6765630000ba8aefce03e88343a51e62b3c1ab', 'width': 640}, {'height': 300, 'url': 'https://i.scdn.co/image/a...",False,sv,[sv],Trailer: Naket och Nära - Ska jag vänta på ”karlhelvetet”?,2021-06-28,day,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...","{'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CA', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE', 'EC', 'EE', 'SV', 'FI', 'FR', 'GR', '...",False,episode,spotify:episode:3cCPgGLmgJqp8k3uQvUfD7,<NA>,<NA>
2,False,https://podz-content.spotifycdn.com/audio/clips/0Tzqb7dzhbJhMiaahTBXqk/clip_0_27088.mp3,Spotify Dok samlar Sveriges starkaste dokumentärer. Nya avsnitt varje torsdag. Learn more about your ad choices. Visit podcastchoices.com/adchoices,27088,True,{'spotify': 'https://open.spotify.com/episode/2zT28jtBbq3k9h5vGEAm6M'},https://api.spotify.com/v1/episodes/2zT28jtBbq3k9h5vGEAm6M,"<p>Spotify Dok samlar Sveriges starkaste dokumentärer. Nya avsnitt varje torsdag.</p><p> </p><p>Learn more about your ad choices. Visit <a href=""https://pod...",2zT28jtBbq3k9h5vGEAm6M,"[{'height': 640, 'url': 'https://i.scdn.co/image/ab6765630000ba8a4cfb2837283ee46f3c9bdb37', 'width': 640}, {'height': 300, 'url': 'https://i.scdn.co/image/a...",False,sv,[sv],Spotify Dok - Sveriges starkaste dokumentärer,2019-07-31,day,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, CY, CZ, DK, DO, DE, EC, EE, SV, FI, FR, GR, GT, HN, HK, HU, IS, IE, IT, LV, LT, LU, MY, MT, MX, NL, NZ, NI, NO,...","{'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CA', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE', 'EC', 'EE', 'SV', 'FI', 'FR', 'GR', '...",False,episode,spotify:episode:2zT28jtBbq3k9h5vGEAm6M,<NA>,<NA>


### `spotify-metadata/spotify_shows.jsonl.zst`

Nao consegui ler amostra do JSONL.ZST. python=No module named 'zstandard'; cli=executavel zstd nao encontrado; duckdb=Invalid Input Error: "maximum_object_size" of 16777216 bytes exceeded while reading file "D:/Mestrado/music-search-engine/data/spotify-metadata/spotify_shows.jsonl.zst" (>21180427 bytes).
 Try increasing "maximum_object_size".

LINE 1: SELECT * FROM read_json_auto('D:/Mestrado/music-search-engine/data/spotify...
                      ^


## 10. Consultas uteis para investigacao

Algumas consultas prontas para responder perguntas comuns sobre o dataset.

In [13]:
# Tracks mais populares na amostra inicial da tabela de tracks
tracks_path = paths["tracks"]

display(
    con.execute(
        f"""
        SELECT id, name, popularity, duration_ms, explicit
        FROM read_parquet('{sql_path(tracks_path)}')
        WHERE popularity IS NOT NULL
        ORDER BY popularity DESC
        LIMIT 20
        """
    ).fetchdf()
)

,id,name,popularity,duration_ms,explicit
0,2plbrEY59IikOBgBGLjaoe,Die With A Smile,100,251667,0
1,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,98,210373,0
2,3sK8wGT43QFpWrvNQsrQya,DtMF,98,237117,1
3,0zirWZTcXBBwGsevrsIpvT,Clean Baby Sleep White Noise (Loopable),97,142222,0
4,7ne4VBA60CxGM75vw0EYad,That’s So True,96,166300,1
5,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,96,274192,1
6,2lTm559tuIvatlT1u0JYG2,BAILE INoLVIDABLE,96,367725,1
7,5vNRhkKd0yEAg8suGBpjeY,APT.,95,169917,0
8,3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,95,261466,0
9,2262bWmqomIaJXwCRHr13j,Sailor Song,95,211978,0


In [14]:
# Generos mais frequentes ligados a artistas
artist_genres_path = paths["artist_genres"]

display(
    con.execute(
        f"""
        SELECT genre, count(*) AS artistas
        FROM read_parquet('{sql_path(artist_genres_path)}')
        GROUP BY genre
        ORDER BY artistas DESC
        LIMIT 30
        """
    ).fetchdf()
)

,genre,artistas
0,opera,22822
1,choral,21188
2,chamber music,20581
3,psytrance,18352
4,rockabilly,14496
5,black metal,14376
6,avant-garde,13802
7,bhajan,13652
8,traditional music,13493
9,dancehall,12724


In [15]:
# Distribuicao de atributos de audio em uma amostra ordenada por leitura fisica
audio_path = paths["audio_features"]

audio_sample = con.execute(
    f"""
    SELECT danceability, energy, valence, acousticness, instrumentalness, speechiness, liveness, tempo, loudness
    FROM read_parquet('{sql_path(audio_path)}')
    LIMIT 50000
    """
).fetchdf()

display(audio_sample.describe().T)

,count,unique,top,freq
danceability,49931,1061,0.68,134
energy,49931,1942,0.534,97
valence,49931,1728,0.961,154
acousticness,49931,4637,0.995,151
instrumentalness,49931,4989,0.0,11453
speechiness,49931,1535,0.0346,149
liveness,49931,1605,0.111,862
tempo,49931,29504,0,41
loudness,49931,18329,-7.434,14


## 11. Proximos passos sugeridos

- Escolher quais tabelas entram no mecanismo de busca: geralmente `tracks`, `artists`, `albums`, `artist_genres` e `track_audio_features`.
- Materializar uma tabela plana menor com apenas colunas necessarias para ranking e exibicao.
- Persistir essa tabela derivada em Parquet ou DuckDB para acelerar experimentos.
- Validar nulos, duplicatas e campos textuais antes de indexar.